# Sentiment Labeling Pipeline

**Steps:**

1. Load `datasets/final/mda_shared_preprocessed.parquet`, explode `sentences`
2. Drop duplicate sentences, then create a randomized **400-sentence** manual sample with positive/negative/neutral coverage
3. Sample **50k sentences** → LLM few-shot labels (inter-rater check; reuses cached results when available)
4. Merge with your manual labels → Cohen's Kappa
   - κ > 0.7 → proceed; else revise prompt in Section 3
5. Label **entire dataset** with LLM
6. Check class proportions (>80% dominant → prompt-refinement placeholder)
7. Spot-check 100 random sentences → Excel
8. Save final labeled dataset

**Labeling guidelines:**

- Clear growth signal / positive factual outcome → **positive**
- Negative factual outcome → **negative**
- Forward-looking statements → **neutral**
- Backward-looking negative statements → **negative**
- Otherwise → **neutral**


## 0. Imports & Config


In [9]:
# Fast dependency install via uv into a repo-local package directory.
import shutil
import subprocess
import sys
from pathlib import Path

PACKAGE_DIR = Path.cwd().resolve() / ".uv_packages"
PACKAGE_DIR.mkdir(exist_ok=True)

if shutil.which("uv") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "uv", "-q"])

subprocess.check_call(
    [
        "uv",
        "pip",
        "install",
        "--target",
        str(PACKAGE_DIR),
        "--no-cache",
        "--upgrade",
        "numpy==1.26.4",
        "pandas==2.2.3",
        "pyarrow>=15,<19",
        "openpyxl",
        "typing_extensions>=4.12.2",
        "pydantic>=2.7",
        "pydantic-core>=2.18",
        "tqdm",
        "vllm",
    ]
)

sys.path.insert(0, str(PACKAGE_DIR))
print(f"Using repo-local package directory: {PACKAGE_DIR}")
print("If this cell changed numpy/pandas, restart the kernel before running imports.")


Using CPython 3.11.11 interpreter at: jupyterlab-venv-pytorch-240/bin/python3


Using repo-local package directory: /common/home/users/w/wlcheng.2023/.uv_packages
If this cell changed numpy/pandas, restart the kernel before running imports.


Resolved 171 packages in 1.43s
Checked 171 packages in 3ms


In [10]:
import ast
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from tqdm.auto import tqdm
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# ── Paths ────────────────────────────────────────────────────────────────────────────────
DATA_DIR = Path.home() / "sentiment_analysis_data"
DATA_PATH = DATA_DIR / "mda_shared_preprocessed.parquet"
MANUAL_OUT = DATA_DIR / "manual_labeling_sample.xlsx"
# Git-tracked file co-located with this notebook (sentiment_analysis/).
# Assumes CWD is the notebook's own directory when running on the cluster.
MANUAL_LABELED_PATH = Path.cwd() / "manual_labeling_sample_LABELED_COMPLETE.xlsx"
FINAL_OUT = DATA_DIR / "labeled_dataset.csv"
SPOT_CHECK_OUT = DATA_DIR / "spot_check_100.xlsx"

# ── Constants ─────────────────────────────────────────────────────────────────────────────
VALID_LABELS = {"positive", "negative", "neutral"}
SEED = 42
NUM_GPUS = (
    len(os.environ["CUDA_VISIBLE_DEVICES"].split(","))
    if os.environ.get("CUDA_VISIBLE_DEVICES")
    else 1
)
CHUNK_SIZE = 8_192
random.seed(SEED)
np.random.seed(SEED)


def read_label_table(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.suffix.lower() == ".csv" else pd.read_excel(path)


## 1. Load Parquet & Explode Sentences


In [11]:
df_raw = pd.read_parquet(DATA_PATH)
print(f"✅ Loaded: {len(df_raw):,} filings")

df = (
    df_raw[
        [
            "doc_id",
            "company_name",
            "filing_type",
            "filing_date",
            "year",
            "quarter",
            "sentences",
        ]
    ]
    .explode("sentences")
    .rename(columns={"sentences": "sentence"})
    .dropna(subset=["sentence"])
    .reset_index(drop=True)
)
df["sentence"] = df["sentence"].astype(str).str.strip()
df = df[df["sentence"].str.len() > 20].reset_index(drop=True)
print(f"✅ Sentences: {len(df):,}")
df.head(3)


✅ Loaded: 18,169 filings
✅ Sentences: 1,251,937


,doc_id,company_name,filing_type,filing_date,year,quarter,sentence
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,presently the companys product line includes a...
1,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,approximately NUM of all sales were generated ...
2,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,on an ongoing basis we re-evaluate our judgmen...


## 2. Deduplicate + Randomized 400-Sentence Sample (with Class Coverage)


In [12]:
dup_counts = df.duplicated(subset=["sentence"], keep=False).sum()
print(f"Total rows: {len(df):,} | Duplicate rows: {dup_counts:,}")

if dup_counts:
    top5 = (
        df[df.duplicated(subset=["sentence"], keep=False)]
        .groupby("sentence")
        .size()
        .nlargest(5)
    )
    for sent, cnt in top5.items():
        print(f"  {cnt}x | {sent[:120]}{'...' if len(sent) > 120 else ''}")


Total rows: 1,251,937 | Duplicate rows: 1,003,558
  1140x | actual results may differ from these estimates under different assumptions or conditions.
  1016x | we evaluate our estimates and assumptions on an ongoing basis.
  652x | our actual results may differ from these estimates under different assumptions or conditions.
  598x | we may be required to seek additional equity or debt financing.
  532x | on an ongoing basis we evaluate our estimates and assumptions.


In [13]:
MANUAL_N = 400

POS_HINTS = (
    "increased",
    "increase",
    "improved",
    "improvement",
    "growth",
    "grew",
    "strong",
    "record",
    "higher",
    "gain",
    "expanded",
    "expansion",
    "profit",
    "profitable",
    "beat",
    "outperformed",
)
NEG_HINTS = (
    "decreased",
    "decrease",
    "declined",
    "decline",
    "lower",
    "loss",
    "impairment",
    "charge",
    "charges",
    "restructuring",
    "adverse",
    "weak",
    "headwind",
    "downturn",
    "reduced",
    "fell",
    "drop",
)
NEUTRAL_HINTS = (
    "expect",
    "expects",
    "expected",
    "anticipate",
    "may",
    "might",
    "could",
    "will",
    "plan",
    "plans",
    "guidance",
    "outlook",
    "forecast",
)


def heuristic_bucket(s: str) -> str:
    s = s.lower()
    if any(w in s for w in NEUTRAL_HINTS):
        return "neutral"
    pos = any(w in s for w in POS_HINTS)
    neg = any(w in s for w in NEG_HINTS)
    if pos and not neg:
        return "positive"
    if neg and not pos:
        return "negative"
    return "neutral"


before = len(df)
df = df.drop_duplicates(subset=["sentence"]).reset_index(drop=True)
print(f"✅ Removed {before - len(df):,} duplicates → {len(df):,} unique sentences")

MANUAL_LABELED_PATH = resolve_manual_labeled_path()
print(f"✅ Using labeled sample: {MANUAL_LABELED_PATH}")

if MANUAL_LABELED_PATH.exists():
    manual_sample = read_label_table(MANUAL_LABELED_PATH).copy()
    if "sample_bucket" not in manual_sample.columns:
        manual_sample["sample_bucket"] = manual_sample["sentence"].apply(
            heuristic_bucket
        )
    manual_sample["manual_label"] = (
        manual_sample["manual_label"].fillna("").astype(str).str.strip().str.lower()
    )
else:
    df["sample_bucket"] = df["sentence"].apply(heuristic_bucket)
    target = MANUAL_N // 3
    parts = [
        grp.sample(n=min(target, len(grp)), random_state=SEED)
        for lbl, grp in df.groupby("sample_bucket")
    ]
    manual_sample = pd.concat(parts, ignore_index=True)
    shortfall = MANUAL_N - len(manual_sample)
    if shortfall > 0:
        rest = df[~df.index.isin(manual_sample.index)]
        manual_sample = pd.concat(
            [
                manual_sample,
                rest.sample(n=min(shortfall, len(rest)), random_state=SEED),
            ],
            ignore_index=True,
        )
    manual_sample = manual_sample.sample(frac=1, random_state=SEED).reset_index(
        drop=True
    )
    manual_sample["manual_label"] = ""
    manual_sample.drop(columns=["sample_bucket"]).to_excel(MANUAL_OUT, index=False)
    print(f"✅ Manual sample written → {MANUAL_OUT}")
    print(manual_sample["sample_bucket"].value_counts().to_string())

manual_sample[
    ["sentence", "sample_bucket", "company_name", "year", "manual_label"]
].head(8)


✅ Removed 799,547 duplicates → 452,390 unique sentences
✅ Using labeled sample: /common/home/users/w/wlcheng.2023/sentiment_analysis_data/manual_labeling_sample_LABELED_COMPLETE.xlsx


,sentence,sample_bucket,company_name,year,manual_label
0,net income was reduced in NUM due to NUM of af...,negative,SailPoint,2014,negative
1,jabil produces our printers to our design spec...,neutral,Zebra_Technologies,2011,neutral
2,the net deferred tax liability from the acquis...,positive,Salesforce,2013,positive
3,we buy xol so that the probability of losses f...,negative,Hippo,2025,negative
4,the increase in interest income and other net ...,positive,KLA,2011,positive
5,these employees are automatically highlighted ...,positive,Genpact,2015,neutral
6,NUM acquisition-related costs represent fees a...,neutral,Sabre,2021,neutral
7,the increase was primarily driven by the trans...,positive,Lemonade,2024,neutral


## 3. LLM Label Manual Sample (Inter-rater Check)


In [14]:
SYSTEM_PROMPT = (
    "You are a financial sentiment classifier for SEC 10-K/10-Q filings. "
    "Return exactly one label: positive, negative, or neutral. "
    "Follow this order of rules: first check whether the sentence is mixed, cut off, incomplete, or ambiguous; "
    "if so, label it neutral. "
    "Judge sentiment by economic effect on the firm, not by tone. "
    "positive = proven or clearly implied expansion or improvement, such as revenue growth, market share gain, "
    "demand growth, profit growth, stronger operating results, or other clear upside. "
    "negative = proven or clearly implied deterioration, such as lower sales or revenue, lower margins, margin compression, "
    "losses, wage inflation, liabilities to the company, impairments, charges, production or supply-chain constraints, "
    "higher costs that hurt the company, or any negative outlook that may affect operations or revenues. "
    "neutral = descriptive, boilerplate, accounting allocation, policy/process text, growth-related expense increases "
    "(for example R&D or employee-benefit spending for growth), or unclear direction. "
    "Critical rule: hypothetical risk language alone (may/could/might/possible) is neutral unless the sentence also states "
    "a concrete negative business impact. "
    "Critical rule: if a sentence contains proven growth that is only partially offset by higher costs, focus on the proven growth. "
    "Critical rule: if the sentence is truly mixed and no dominant direction is clear, label neutral. "
    "Respond with only the label word."
)

FEW_SHOT = (
    'Sentence: "Revenue increased 18% driven by strong demand." Label: positive\n'
    'Sentence: "Our market share expanded and customer adoption accelerated." Label: positive\n'
    'Sentence: "Net income declined 32% due to higher costs." Label: negative\n'
    'Sentence: "Gross margins were lower because of pricing pressure." Label: negative\n'
    'Sentence: "We may face increased competition." Label: neutral\n'
    'Sentence: "We increased R&D and employee benefits to support growth." Label: neutral\n'
    'Sentence: "Software development expenses declined by $12 million." Label: negative\n'
    'Sentence: "Nursing wage inflation increased during the quarter." Label: negative\n'
    'Sentence: "The yield earned on interest-earning assets decreased six basis points." Label: negative\n'
    'Sentence: "Travel bookings declined, partially offset by lower cancellations." Label: negative\n'
    'Sentence: "These increases were partially offset by losses on foreign currency transactions." Label: neutral\n'
    'Sentence: "Atos and began reporting it as a discontinued operation." Label: neutral\n'
)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
if "llm" not in globals():
    llm = LLM(
        model=MODEL_ID,
        quantization="awq",
        max_model_len=1024,
        tensor_parallel_size=NUM_GPUS,
        gpu_memory_utilization=0.95,
        enable_prefix_caching=True,
    )
if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

sampling_params = SamplingParams(temperature=0, max_tokens=5)


def _build_prompt(sentence: str) -> str:
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": FEW_SHOT + f'Sentence: "{sentence}" Label:'},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )


def llm_label_batch(sentences: list[str]) -> list[str]:
    prompts = [_build_prompt(s) for s in sentences]
    raw = []
    print(f"  Sending {len(prompts):,} sentences to LLM")
    for start in tqdm(
        range(0, len(prompts), CHUNK_SIZE), desc="LLM labeling", unit="chunk"
    ):
        raw.extend(llm.generate(prompts[start : start + CHUNK_SIZE], sampling_params))

    output = []
    for out in raw:
        text = out.outputs[0].text.strip().lower()
        output.append(text if text in VALID_LABELS else "neutral")

    print(f"  Done: {pd.Series(output).value_counts().to_dict()}")
    return output


print(f"✅ LLM ready on {NUM_GPUS} GPU(s)")
print(f"✅ Labeling {len(manual_sample):,} manual sentences...")
llm_manual = manual_sample[["sentence"]].copy()
llm_manual["llm_label"] = llm_label_batch(manual_sample["sentence"].tolist())
print(llm_manual["llm_label"].value_counts().to_string())

✅ LLM ready on 1 GPU(s)
✅ Labeling 400 manual sentences...
  Sending 400 sentences to LLM


LLM labeling:   0%|          | 0/1 [00:00<?, ?chunk/s]

Adding requests:   0%|          | 0/400 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/400 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  Done: {'neutral': 279, 'positive': 67, 'negative': 54}
llm_label
neutral     279
positive     67
negative     54


## 4. Merge Manual Labels & Compute Cohen's Kappa

> **Action required before running this cell:**  
> Open `manual_labeling_sample.xlsx`, fill the `manual_label` column  
> (values: `positive` / `negative` / `neutral`), save as  
> `manual_labeling_sample_LABELED_COMPLETE.xlsx` in the same folder.


In [15]:
manual_labeled = read_label_table(MANUAL_LABELED_PATH)
manual_labeled["manual_label"] = (
    manual_labeled["manual_label"].fillna("").astype(str).str.strip().str.lower()
)
manual_labeled = manual_labeled[
    manual_labeled["manual_label"].isin(VALID_LABELS)
].copy()

merged = manual_labeled.merge(
    llm_manual[["sentence", "llm_label"]], on="sentence", how="inner"
)
print(f"✅ Matched for kappa: {len(merged):,} rows")

kappa = cohen_kappa_score(merged["manual_label"], merged["llm_label"])
print(f"\n✅ Cohen's Kappa = {kappa:.4f}")
print(
    "✅ Good agreement — proceed."
    if kappa > 0.7
    else "⚠️  Poor agreement — revise prompt in Section 3 and re-run."
)

pd.crosstab(
    merged["manual_label"],
    merged["llm_label"],
    rownames=["manual"],
    colnames=["llm"],
    margins=True,
)


✅ Matched for kappa: 400 rows

✅ Cohen's Kappa = 0.6140
⚠️  Poor agreement — revise prompt in Section 3 and re-run.


llm,negative,neutral,positive,All
manual,,,,
negative,43,23,2,68
neutral,11,233,17,261
positive,0,23,48,71
All,54,279,67,400


In [16]:
disagreed = merged[merged["manual_label"] != merged["llm_label"]].copy()
print(f"❌ Disagreed rows: {len(disagreed):,} / {len(merged):,}")

cols = [
    c
    for c in [
        "sentence",
        "manual_label",
        "llm_label",
        "company_name",
        "year",
        "quarter",
        "filing_date",
    ]
    if c in disagreed.columns
]
disagreed = (
    disagreed[cols].sort_values(["manual_label", "llm_label"]).reset_index(drop=True)
)

pair_counts = (
    disagreed.groupby(["manual_label", "llm_label"]).size().sort_values(ascending=False)
)
print("\nTop disagreement pairs (manual -> llm):")
print(pair_counts.head(10).to_string())

# Save for prompt revision workflow.
out_path = DATA_DIR / "manual_vs_llm_disagreements.csv"
disagreed.to_csv(out_path, index=False)
print(f"\n✅ Saved disagreement table → {out_path}")

print("\nPreview (first 20 disagreements):")
preview_cols = [
    c for c in ["manual_label", "llm_label", "sentence"] if c in disagreed.columns
]
print(disagreed[preview_cols].head(20).to_string(index=False, max_colwidth=140))

disagreed.head(30)

❌ Disagreed rows: 76 / 400

Top disagreement pairs (manual -> llm):
manual_label  llm_label
negative      neutral      23
positive      neutral      23
neutral       positive     17
              negative     11
negative      positive      2

✅ Saved disagreement table → /common/home/users/w/wlcheng.2023/sentiment_analysis_data/manual_vs_llm_disagreements.csv

Preview (first 20 disagreements):
manual_label llm_label                                                                                                                                     sentence
    negative   neutral we buy xol so that the probability of losses from a single occurrence exceeding the protection purchased is no more than NUM or equivalen...
    negative   neutral prepaid expenses and other current assets increased primarily due to higher prepaid employee benefit expenses largely associated with rec...
    negative   neutral amortization expense increased for the three months ended march NUM NUM as compared to t

,sentence,manual_label,llm_label,company_name,year,quarter,filing_date
0,we buy xol so that the probability of losses f...,negative,neutral,Hippo,2025,Q1,2025-03-06
1,prepaid expenses and other current assets incr...,negative,neutral,Adobe,2015,Q1,2015-03-25
2,amortization expense increased for the three m...,negative,neutral,TechTarget,2022,Q2,2022-05-10
3,software development expenses declined by NUM ...,negative,neutral,Blackboxstocks,2025,Q1,2025-03-21
4,the lower gross margins on product revenues we...,negative,neutral,NetApp,2015,Q1,2015-02-25
5,the increase in r d expenses was primarily due...,negative,neutral,Aehr_Test_Systems,2022,Q1,2022-01-14
6,nursing wage inflation increased NUM while non...,negative,neutral,Gen_Digital,2019,Q2,2019-05-10
7,the decrease was partially offset by a decreas...,negative,neutral,Veradigm,2021,Q4,2021-11-05
8,the decline in average selling prices was prim...,negative,neutral,Supermicro,2020,Q1,2020-02-07
9,for each of the quarters ended september NUM N...,negative,neutral,PAR_Technology,2019,Q4,2019-11-08


## 5. Label Entire Dataset

Runs only if kappa > 0.7.


In [ ]:
assert kappa > 0.7, f"Kappa = {kappa:.4f} ≤ 0.7. Revise prompt before proceeding."

df_full = df.copy()
print(f"✅ Labeling {len(df_full):,} sentences...")
df_full["llm_label"] = llm_label_batch(df_full["sentence"].tolist())
print("\n✅ Label distribution:")
print(df_full["llm_label"].value_counts().to_string())


## 6. Check Class Proportions


In [ ]:
props = df_full["llm_label"].value_counts(normalize=True)
print("✅ Class proportions:")
print(props.map("{:.2%}".format).to_string())

dom_cls = props.idxmax()
dom_pct = props.max()

if dom_pct > 0.80:
    print(f"\n⚠️  '{dom_cls}' dominates at {dom_pct:.1%}. Consider:")
    print("  1. Add more contrastive few-shot examples for minority classes")
    print("  2. Tighten/loosen the dominant-class definition in the prompt")
    print("  3. Re-run Section 3 with the revised prompt and re-check kappa")
else:
    print(f"\n✅ Class distribution looks healthy (no class > 80%).")


## 7. Spot-Check 100 Random Sentences


In [ ]:
spot = df_full.sample(n=100, random_state=SEED)[
    ["sentence", "company_name", "year", "filing_type", "llm_label"]
].reset_index(drop=True)
spot["reviewer_label"] = ""  # optional: fill in to verify quality

spot.to_excel(SPOT_CHECK_OUT, index=False)
print(f"✅ Spot-check sample saved to '{SPOT_CHECK_OUT}'")
spot.head(10)


## 8. Save Final Labeled Dataset


In [ ]:
df_final = df_full[
    [
        "doc_id",
        "company_name",
        "filing_type",
        "filing_date",
        "year",
        "quarter",
        "sentence",
        "llm_label",
    ]
].rename(columns={"llm_label": "sentiment"})
df_final.to_csv(FINAL_OUT, index=False)

print(f"✅ Final dataset saved -> {FINAL_OUT}")
print(f"✅ Shape: {df_final.shape}")
print("\n✅ Label distribution:")
print(df_final["sentiment"].value_counts().to_string())
df_final.head(5)